#  Evaluating Sustainability using EvaluationAgent

This tutorial demonstrates how to use the `pruna` package to evaluate the sustainability of a model. Image generation is among the most energy-intensive inference workloads. Compression methods promise to reduce this, but their savings vary with hardware and workload, so the only way to know what a method actually saves *for you* is to measure it. Sustainability benchmarks make those savings visible and comparable.

We will use the `stable-diffusion-v1-5` model and a subset of the `LAION256` dataset, comparing a baseline against two optimization methods (`deepcache` and `torch_compile`). Any execution times given below are measured on a T4 GPU.

In [ ]:
# if you are not running the latest version of this tutorial, make sure to install the matching version of pruna
# the following command will install the latest version of pruna
%pip install pruna

### 1. Loading the model

First, load your model.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

from pruna.engine.pruna_model import PrunaModel

pipe = AutoPipelineForText2Image.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe = pipe.to("cuda")
model = PrunaModel(pipe)
pipe.set_progress_bar_config(disable=True)

# Shared generation parameters, applied to every configuration so results are comparable
GEN_ARGS = {"num_inference_steps": 25, "guidance_scale": 7.5}

### 2. Benchmark metrics

`pruna` tracks three sustainability metrics:

- **`total_time`**: wall-clock time to run the benchmark iterations (ms)
- **`energy_consumed`**: total energy drawn during inference (kWh)
- **`co2_emissions`**: estimated CO2-equivalent emissions based on energy consumed and hardware location (kg)

We also include **`clip_score`**, a quality metric measuring how well generated images match their prompts, so we can check whether efficiency gains come at a quality cost.

We will pass these metrics to the evaluation `Task` as a list of metric instances. For other ways to specify metrics, see the [evaluation documentation](https://docs.pruna.ai/en/stable/docs_pruna/user_manual/evaluate.html).

In [ ]:
from pruna.evaluation.metrics import (
    CO2EmissionsMetric,
    EnergyConsumedMetric,
    TorchMetricWrapper,
    TotalTimeMetric,
)

request = [
    TotalTimeMetric(n_iterations=10, n_warmup_iterations=3),
    EnergyConsumedMetric(n_iterations=10, n_warmup_iterations=3),
    CO2EmissionsMetric(n_iterations=10, n_warmup_iterations=3),
    TorchMetricWrapper("clip_score"),
]

### 3. Create an EvaluationAgent and a Task

Pruna's evaluation process uses a Task to define which metrics to calculate and provide the evaluation data. The EvaluationAgent then takes this Task and handles running the model inference, passing the inputs, ground truth, and predictions to each metric, and collecting the results.




In [ ]:
from pruna.data.pruna_datamodule import PrunaDataModule
from pruna.evaluation.evaluation_agent import EvaluationAgent
from pruna.evaluation.task import Task

datamodule = PrunaDataModule.from_string("LAION256")
datamodule.limit_datasets(10)  # Quality metrics run over these 10 samples. Timing metrics benchmark a single batch
task = Task(request, datamodule)
eval_agent = EvaluationAgent(task)

### 4. Evaluate the baseline model

We can evaluate a model by calling the `evaluate` method of the EvaluationAgent.

In [ ]:
model.inference_handler.model_args.update(GEN_ARGS)

base_results = eval_agent.evaluate(model)
for r in base_results:
    print(f"{r.name}: {r.result:.4g}")

### 5. Smash the model with DeepCache

In [ ]:
import copy

from pruna import smash
from pruna.config.smash_config import SmashConfig
from pruna.engine.utils import safe_memory_cleanup

smash_config = SmashConfig()
smash_config.add(dict(deepcache=True))

pipe = pipe.to("cpu")
safe_memory_cleanup()

copy_pipe = copy.deepcopy(pipe).to("cuda")
smashed_pipe = smash(copy_pipe, smash_config)
smashed_pipe.set_progress_bar_config(disable=True)
smashed_pipe.inference_handler.model_args.update(GEN_ARGS)

### 6. Evaluate the first smashed model (DeepCache)

We now evaluate the smashed model by calling evaluate again.

In [ ]:
smashed_results = eval_agent.evaluate(smashed_pipe)
for r in smashed_results:
    print(f"{r.name}: {r.result:.4g}")

### 7. Analyze baseline vs DeepCache

With both configurations evaluated on the same task, we can now compare them side by side. For time, energy, and CO2 a negative change is an improvement. For CLIP score, higher is better.

In [ ]:
import matplotlib.pyplot as plt

base = {r.name: r.result for r in base_results}
smashed = {r.name: r.result for r in smashed_results}

metrics = {
    "Inference time (s)": ("total_time", 1e-3),   # ms -> s
    "Energy (Wh)": ("energy_consumed", 1e3),      # kWh -> Wh
    "CO2 emissions (g)": ("co2_emissions", 1e3),  # kg -> g
    "CLIP score": ("clip_score", 1),
}

fig, axes = plt.subplots(1, 4, figsize=(15, 4), layout="constrained")

for ax, (label, (key, scale)) in zip(axes, metrics.items()):
    values = [base[key] * scale, smashed[key] * scale]
    bars = ax.bar(["Baseline", "DeepCache"], values, color=["#8a8a8a", "#7c3aed"])
    change = (values[1] / values[0] - 1) * 100
    ax.bar_label(bars, labels=[f"{values[0]:.3f}", f"{values[1]:.3f}\n({change:+.1f}%)"], fontsize=9)
    ax.set_ylim(0, max(values) * 1.3)
    ax.set_title(label)

fig.suptitle("Benchmark metrics: baseline vs. DeepCache")
plt.show()

### 8. Smash the model with torch_compile

In [ ]:
# Free the DeepCache pipeline before building the next configuration
del smashed_pipe
safe_memory_cleanup()

# Smash with torch.compile (start from a fresh copy of the original pipe)
compile_config = SmashConfig()
compile_config.add(dict(torch_compile=True))

compile_pipe = smash(copy.deepcopy(pipe).to("cuda"), compile_config)
compile_pipe.set_progress_bar_config(disable=True)
compile_pipe.inference_handler.model_args.update(GEN_ARGS)

### 9. Evaluate the second smashed model (torch_compile)

In [ ]:
compiled_results = eval_agent.evaluate(compile_pipe)
for r in compiled_results:
    print(f"{r.name}: {r.result:.4g}")

### 10. Analyze baseline vs DeepCache vs torch_compile

With all three configurations evaluated on the same task, we can compare them in a single visual.


In [ ]:
all_results = {
    "Baseline": base_results,
    "DeepCache": smashed_results,
    "torch.compile": compiled_results,
}
data = {name: {r.name: r.result for r in res} for name, res in all_results.items()}

# Convert to more readable units for small workloads
metrics = {
    "Inference time (s)": ("total_time", 1e-3),   # ms -> s
    "Energy (Wh)": ("energy_consumed", 1e3),      # kWh -> Wh
    "CO2 emissions (g)": ("co2_emissions", 1e3),  # kg -> g
    "CLIP score": ("clip_score", 1),
}

names = list(all_results)
colors = ["#8a8a8a", "#7c3aed", "#0d9488"]

fig, axes = plt.subplots(1, 4, figsize=(15, 4), layout="constrained")

for ax, (label, (key, scale)) in zip(axes, metrics.items()):
    vals = [data[name][key] * scale for name in names]
    bars = ax.bar(names, vals, color=colors)
    labels = [f"{v:.3f}" if i == 0 else f"{v:.3f}\n({(v / vals[0] - 1) * 100:+.1f}%)"
              for i, v in enumerate(vals)]
    ax.bar_label(bars, labels=labels, fontsize=9)
    ax.set_ylim(0, max(vals) * 1.3)
    ax.set_title(label)

fig.suptitle("Benchmark metrics vs. baseline")
plt.show()

### 11. Takeaway

All three sustainability metrics move together: less inference time means proportionally less energy and CO2. DeepCache achieves this by skipping redundant UNet computation, while torch.compile runs the same computation with faster kernels, so its gains depend on the hardware.

CLIP score shows a slight drop for DeepCache, hinting at a small quality tradeoff even in this limited run. For a stricter quality comparison, see the [CMMD evaluation tutorial](./evaluation_agent_cmmd.ipynb).